# NB_00 — Project Setup

**Purpose:** Mount Google Drive, create the canonical project directory, and download
only the data artefacts needed to continue from Stage 5 (LoRA fine-tuning) onward.
Stages 1–4 were completed on AWS; their outputs (the JSONL splits and LoRA adapter)
are what we need here.

Run this notebook **once** before any other notebook in this series.

---
| Notebook | Stage | Needs GPU? |
|---|---|---|
| NB_00_setup | Project setup / data download | No |
| NB_05_finetuning | Stage 3: LoRA fine-tuning | Yes (A100) |
| NB_06_evaluation | Stage 4: Evaluation | Yes (A100) |
| NB_07_downstream_nlp | Stage 6: NER + POS tagging | No (CPU OK) |
| NB_08_baseline | Stage 7: TrOCR baseline | Yes (A100) |

## Step 0.1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


## Step 0.2 — Define project root and create directory structure

All notebooks in this series use `PROJECT_ROOT` as their single source of truth for paths.

In [2]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project'

dirs = [
    'data/train',
    'data/eval',
    'data/trocr_images/train',
    'data/trocr_images/eval',
    'models/lora_adapter/run-2',
    'camel_data',
    'logs/run-1',
    'logs/run-2',
    'logs/stage6',
    'logs/stage7',
]

for d in dirs:
    os.makedirs(f'{PROJECT_ROOT}/{d}', exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print('Directory structure created.')

Project root: /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project
Directory structure created.


## Step 0.3 — Install git-lfs and clone only the necessary artefacts from the repo

We use a **sparse checkout** so we do not pull the full model weights (which are large and
will be re-downloaded from Hugging Face at training time). We only need:
- `data/train/train.jsonl` — 1,120-sample training split
- `data/eval/eval.jsonl` — 280-sample eval split

The LoRA adapter from Run 2 is NOT copied from the repo; it will be produced fresh by
NB_05_finetuning. If you already have it on Drive from a previous run, skip Step 0.4.

In [3]:
import subprocess, shutil, os

REPO_URL  = 'https://github.com/thtahamid/nlp_project.git'
CLONE_DIR = '/content/nlp_project_clone'  # temp clone, NOT inside Drive

# ── 1. Install git-lfs ────────────────────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', 'git-lfs'], capture_output=True, check=True)
subprocess.run(['git', 'lfs', 'install'], capture_output=True, check=True)

# ── 2. Sparse clone (no LFS blobs, only the two JSONL files) ─────────────
if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

subprocess.run([
    'git', 'clone',
    '--filter=blob:none',   # skip large blobs
    '--no-checkout',
    REPO_URL, CLONE_DIR
], check=True)

subprocess.run(['git', '-C', CLONE_DIR, 'sparse-checkout', 'init', '--cone'], check=True)
subprocess.run([
    'git', '-C', CLONE_DIR, 'sparse-checkout', 'set',
    'data/train', 'data/eval'
], check=True)
subprocess.run(['git', '-C', CLONE_DIR, 'checkout'], check=True)

print('Sparse clone complete.')

Sparse clone complete.


## Step 0.4 — Copy JSONL splits to project root

In [4]:
import shutil, os

files_to_copy = [
    ('data/train/train.jsonl', f'{PROJECT_ROOT}/data/train/train.jsonl'),
    ('data/eval/eval.jsonl',   f'{PROJECT_ROOT}/data/eval/eval.jsonl'),
]

for src_rel, dst in files_to_copy:
    src = f'{CLONE_DIR}/{src_rel}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Copied {src_rel} → {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'WARNING: {src} not found — check the repo structure.')

# Clean up the temp clone to save Colab disk space
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print('\nTemp clone removed. Setup complete.')

Copied data/train/train.jsonl → /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/data/train/train.jsonl  (20.5 MB)
Copied data/eval/eval.jsonl → /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/data/eval/eval.jsonl  (5.2 MB)

Temp clone removed. Setup complete.


## Step 0.5 — Verify the data files

In [5]:
import json

for split, path in [
    ('train', f'{PROJECT_ROOT}/data/train/train.jsonl'),
    ('eval',  f'{PROJECT_ROOT}/data/eval/eval.jsonl'),
]:
    with open(path) as f:
        lines = f.readlines()
    sample = json.loads(lines[0])
    print(f'{split}: {len(lines)} samples')
    print(f'  Keys in first sample: {list(sample.keys())}')
    roles = [m["role"] for m in sample["messages"]]
    print(f'  Message roles: {roles}')

print('\nSetup verified. You can now run NB_05_finetuning.ipynb.')

train: 1120 samples
  Keys in first sample: ['messages']
  Message roles: ['system', 'user', 'assistant']
eval: 280 samples
  Keys in first sample: ['messages']
  Message roles: ['system', 'user', 'assistant']

Setup verified. You can now run NB_05_finetuning.ipynb.


## Step 0.6 — Install base dependencies (optional, run once per session)

Heavier installs (bitsandbytes, peft, camel-tools) are handled in their respective notebooks.
This cell installs the lightweight utilities used across all notebooks.

In [ ]:
!pip install jiwer Pillow --quiet
print('Base dependencies installed.')